# Notebook 02 — Leakage Taxonomy and Descriptor Protocols — 

**Project:** CMT Path A — Leakage-audited multi-ion computed insertion-electrode benchmark.

**Patch purpose:** tighten P2/P3 leakage rules so that P2 excludes post-DFT energy/electronic/provenance features and P3 excludes target-defining post-DFT features.

This notebook performs **no ML training, no ranking, no CDE matching, no criticality filtering, and no manuscript writing**.

In [ ]:
# ============================================================
# Notebook 02: Leakage taxonomy and descriptor protocols — 
# CMT Path A: Leakage-audited multi-ion computed insertion-electrode benchmark
# ============================================================
#
# Purpose:
#   Use Notebook 01 outputs to:
#     1. define target plausibility masks,
#     2. build a master feature table,
#     3. classify features by battery-specific leakage level,
#     4. define target-specific descriptor protocols P0-P4,
#     5. save feature lists and protocol audit files for Notebook 04.
#
# Strict exclusions in this notebook:
#   - No ML training
#   - No hyperparameter tuning
#   - No model selection
#   - No ranking
#   - No CDE matching
#   - No criticality filtering
#   - No manuscript writing
# ============================================================

from __future__ import annotations

import os
import sys
import re
import json
import math
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

try:
    import importlib.metadata as importlib_metadata
except Exception:
    import importlib_metadata

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# -
# Canonical clean-room input and output namespaces
# -
def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

NB08_DIR = artifact_namespace("01", REPOSITORY_ROOT)
BASE_DIR = artifact_namespace("02", REPOSITORY_ROOT)
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"
PROTOCOL_DIR = PROCESSED_DIR / "protocol_feature_lists"

for d in [PROCESSED_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR, PROTOCOL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

LOG_ROWS = []

def log_event(stage: str, level: str, message: str, extra: dict | None = None):
    row = {
        "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "stage": stage,
        "level": level,
        "message": message,
        "extra_json": json.dumps(extra or {}, default=str),
    }
    LOG_ROWS.append(row)
    if level.upper() in {"WARNING", "ERROR"}:
        print(f"[{level.upper()}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(LOG_ROWS).to_csv(LOG_DIR / "02_event_log.csv", index=False)

def write_json_safe(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str, ensure_ascii=False)

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except Exception:
        return "not_installed_or_unknown"

def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)

def safe_float(x):
    try:
        if x is None or x == "":
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def sanitize_colname(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^0-9A-Za-z_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if re.match(r"^\d", s):
        s = "x_" + s
    return s

log_event("init", "INFO", "Notebook 02  initialized.", {"nb08_dir": str(NB08_DIR)})
print(f"Using Notebook 01 outputs from: {NB08_DIR}")
print(f"Notebook 02 outputs will be saved under: {BASE_DIR}")

In [ ]:
# ============================================================
# Load Notebook 01 outputs
# ============================================================

core_path = NB08_DIR / "processed" / "01_multion_insertion_electrodes_core_with_groups.csv"
summary_path = NB08_DIR / "processed" / "01_linked_materials_summary.csv"
linked_ids_path = NB08_DIR / "processed" / "01_linked_material_ids_long.csv"
gate_path = NB08_DIR / "audit" / "01_dataset_feasibility_gate.csv"
decision_path = NB08_DIR / "metadata" / "01_final_decision.json"

required_paths = [core_path, summary_path, linked_ids_path, gate_path]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required Notebook 01 files:\n" + "\n".join(missing_paths))

core_df = pd.read_csv(core_path, low_memory=False)
summary_df = pd.read_csv(summary_path, low_memory=False)
linked_ids_long_df = pd.read_csv(linked_ids_path, low_memory=False)
gate_df = pd.read_csv(gate_path)

if decision_path.exists():
    with open(decision_path, "r", encoding="utf-8") as f:
        nb08_decision = json.load(f)
else:
    nb08_decision = {}

print("Loaded Notebook 01 files:")
print(f"  core_df:              {core_df.shape}")
print(f"  summary_df:           {summary_df.shape}")
print(f"  linked_ids_long_df:   {linked_ids_long_df.shape}")
print(f"  feasibility gate:     {gate_df.shape}")
print(f"  Notebook 01 decision: {nb08_decision.get('final_decision', 'unknown')}")

REQUIRED_CORE_COLUMNS = [
    "record_index", "working_ion", "battery_formula", "formula_charge", "formula_discharge",
    "framework_formula", "chemsys", "elements", "average_voltage", "capacity_grav",
    "capacity_vol", "energy_grav", "energy_vol", "max_delta_volume", "stability_charge",
    "stability_discharge", "stability_worst", "fracA_charge", "fracA_discharge",
    "num_steps", "max_voltage_step", "id_charge", "id_discharge", "material_ids",
    "framework_formula_reduced", "host_chemsys_no_working_ion", "chemical_system_uid",
    "working_ion_group", "electrode_uid", "framework_uid", "coarse_family",
]

missing_core_columns = [c for c in REQUIRED_CORE_COLUMNS if c not in core_df.columns]
if missing_core_columns:
    raise ValueError(f"Missing required columns in Notebook 01 core file: {missing_core_columns}")

if core_df["electrode_uid"].nunique() != len(core_df):
    log_event(
        "load_inputs", "WARNING",
        "electrode_uid is not unique. Notebook 04 must handle duplicate records carefully.",
        {"n_records": len(core_df), "n_unique_electrode_uid": core_df["electrode_uid"].nunique()},
    )

if "record_source_id" in core_df.columns and core_df["record_source_id"].nunique(dropna=True) < 10:
    log_event(
        "load_inputs", "WARNING",
        "record_source_id has very few unique values and will not be used as an electrode identifier.",
        {"n_unique_record_source_id": int(core_df["record_source_id"].nunique(dropna=True))},
    )

display(core_df.head())
save_event_log()

In [ ]:
# ============================================================
# Environment metadata and no-ML assertion
# ============================================================

software_environment = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "python_version": sys.version,
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pymatgen_version": package_version("pymatgen"),
    "notebook_name": "02_leakage_taxonomy_and_descriptor_protocols.ipynb",
    "notebook_purpose": "Leakage taxonomy and descriptor protocol construction only",
    "patch_status": "P2/P3 leakage rules tightened for post-DFT energy/electronic/provenance features",
}

write_json_safe(software_environment, METADATA_DIR / "02_software_environment.json")

no_ml_assertion = {
    "ml_training_performed": False,
    "model_fitting_performed": False,
    "hyperparameter_tuning_performed": False,
    "ranking_performed": False,
    "cde_matching_performed": False,
    "criticality_filtering_performed": False,
    "manuscript_writing_performed": False,
    "allowed_operations": [
        "target plausibility mask creation",
        "feature table construction",
        "feature leakage taxonomy",
        "target-specific protocol feature-list generation",
        "audit and manifest writing",
    ],
}

write_json_safe(no_ml_assertion, METADATA_DIR / "02_no_ml_assertion.json")

display(pd.DataFrame([software_environment]))
display(pd.DataFrame([no_ml_assertion]))

In [ ]:
# ============================================================
# Target definitions, leakage classes, and protocol definitions — 
# ============================================================

TARGET_COLS = [
    "average_voltage", "capacity_grav", "capacity_vol", "energy_grav", "energy_vol",
    "max_delta_volume", "stability_charge", "stability_discharge", "stability_worst",
]

PRIMARY_BENCHMARK_TARGETS = [
    "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume", "stability_worst",
]

TARGET_UNITS = {
    "average_voltage": "V",
    "capacity_grav": "mAh/g",
    "capacity_vol": "mAh/cm3",
    "energy_grav": "Wh/kg",
    "energy_vol": "Wh/L_or_Wh/dm3_database_dependent",
    "max_delta_volume": "fraction_or_database_convention",
    "stability_charge": "eV/atom",
    "stability_discharge": "eV/atom",
    "stability_worst": "eV/atom",
}

LEAKAGE_CLASS_DEFINITIONS = {
    "L0": {
        "name": "safe descriptor",
        "definition": "No direct target relationship identified by the rule-based battery leakage audit.",
    },
    "L1": {
        "name": "direct target duplicate",
        "definition": "Feature is the exact target or a direct duplicate of the target.",
    },
    "L2": {
        "name": "target-defining component",
        "definition": "Feature is a mathematical component, endpoint quantity, or stored component used to define/reconstruct the target.",
    },
    "L3": {
        "name": "stoichiometric near-target descriptor",
        "definition": "Feature encodes working-ion insertion amount or charge/discharge stoichiometry closely tied to capacity/energy.",
    },
    "L4": {
        "name": "post-DFT adjacent descriptor",
        "definition": "Feature is post-DFT or computed-record adjacent to the target but not a direct/defining duplicate.",
    },
    "L5": {
        "name": "group/domain leakage",
        "definition": "Feature is a group, split, family, chemical-system, or domain label that can leak validation structure.",
    },
}

PROTOCOL_DEFINITIONS = {
    "P0": {
        "name": "full_feature_leaky_baseline_no_direct_target",
        "description": (
            "Historical/full-feature baseline. Direct target duplicate L1 and group/domain L5 are removed, "
            "but target-adjacent L2/L3/L4 descriptors are allowed to quantify optimistic leakage-prone performance."
        ),
        "allowed_leakage_levels": ["L0", "L2", "L3", "L4"],
    },
    "P1": {
        "name": "composition_only_clean",
        "description": (
            "Framework-composition and known working-ion physicochemical descriptors only. "
            "No charge/discharge insertion-window stoichiometry, no stored electrochemical targets, no group labels."
        ),
        "allowed_leakage_levels": ["L0"],
    },
    "P2": {
        "name": "composition_plus_relaxed_structure_clean_strict",
        "description": (
            "P1 plus only conservative relaxed-structure summary descriptors such as nsites, volume, density, "
            "volume per site, and symmetry-number where target-specific leakage is L0. "
            ": P2 explicitly excludes post-DFT energy, hull-stability, electronic, stability-boolean, "
            "and provenance flags such as formation_energy, energy_above_hull, band_gap, is_metal, is_stable, theoretical."
        ),
        "allowed_leakage_levels": ["L0"],
    },
    "P3": {
        "name": "post_DFT_decision_support_clean_target_specific",
        "description": (
            "P2 plus post-DFT decision-support descriptors that are not target-defining for the specific target. "
            ": for voltage/energy, endpoint formation-energy descriptors are excluded; for stability, "
            "endpoint hull/stability descriptors are excluded; for volume-change, endpoint volume/density descriptors are excluded. "
            "P3 must be interpreted as post-DFT decision support, not pre-DFT discovery."
        ),
        "allowed_leakage_levels": ["L0", "L4"],
    },
    "P4": {
        "name": "leakage_stress_test",
        "description": (
            "Intentional leakage stress test. Direct target duplicates and target-adjacent descriptors may be included. "
            "Use only to demonstrate how leakage inflates performance; not a valid predictive protocol."
        ),
        "allowed_leakage_levels": ["L0", "L1", "L2", "L3", "L4"],
    },
}

write_json_safe(
    {
        "targets": TARGET_COLS,
        "primary_benchmark_targets": PRIMARY_BENCHMARK_TARGETS,
        "target_units": TARGET_UNITS,
        "leakage_class_definitions": LEAKAGE_CLASS_DEFINITIONS,
        "protocol_definitions": PROTOCOL_DEFINITIONS,
        "patch_notes": [
            "P2 now excludes formation_energy, energy_above_hull, band_gap, is_metal, is_stable, theoretical.",
            "P3 voltage/energy excludes endpoint formation-energy descriptors.",
            "P3 stability excludes endpoint energy_above_hull/is_stable descriptors.",
            "P3 volume-change excludes endpoint volume/density/delta-volume descriptors.",
        ],
    },
    METADATA_DIR / "02_protocol_definitions.json",
)

display(pd.DataFrame.from_dict(LEAKAGE_CLASS_DEFINITIONS, orient="index"))
display(pd.DataFrame.from_dict(PROTOCOL_DEFINITIONS, orient="index"))

In [ ]:
# ============================================================
# Composition utilities
# ============================================================

try:
    from pymatgen.core import Composition, Element
    PYMATGEN_AVAILABLE = True
except Exception as exc:
    PYMATGEN_AVAILABLE = False
    log_event("pymatgen", "WARNING", "pymatgen is unavailable. Composition descriptors will be reduced.", {"error": str(exc)})

SELECTED_ELEMENTS = [
    "Li", "Na", "K", "O", "F", "P", "S", "Si", "C", "N", "B", "Cl", "Br", "I",
    "Mn", "Fe", "Co", "Ni", "Cu", "V", "Ti", "Cr", "Al", "Mg", "Ca", "Zn", "Ag", "As",
]

TRANSITION_METALS = {
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Y", "Zr", "Nb", "Mo",
    "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
}

def parse_composition(formula):
    fs = safe_str(formula).strip()
    if not fs or not PYMATGEN_AVAILABLE:
        return None
    try:
        return Composition(fs)
    except Exception:
        return None

def element_property(el_symbol: str, prop: str):
    if not PYMATGEN_AVAILABLE:
        return np.nan
    try:
        el = Element(el_symbol)
        val = getattr(el, prop, None)
        if val is None:
            return np.nan
        return float(val)
    except Exception:
        return np.nan

def weighted_stats(values, weights):
    vals = np.array(values, dtype=float)
    w = np.array(weights, dtype=float)
    mask = np.isfinite(vals) & np.isfinite(w) & (w > 0)
    if mask.sum() == 0:
        return {"mean": np.nan, "min": np.nan, "max": np.nan, "range": np.nan, "std": np.nan}
    vals = vals[mask]
    w = w[mask]
    mean = np.average(vals, weights=w)
    var = np.average((vals - mean) ** 2, weights=w)
    return {"mean": float(mean), "min": float(np.min(vals)), "max": float(np.max(vals)), "range": float(np.max(vals) - np.min(vals)), "std": float(np.sqrt(var))}

def formula_composition_features(formula, prefix: str, selected_elements=SELECTED_ELEMENTS):
    out = {}
    comp = parse_composition(formula)
    if comp is None:
        out[f"{prefix}_parse_ok"] = 0
        out[f"{prefix}_n_elements"] = np.nan
        out[f"{prefix}_n_atoms"] = np.nan
        for el in selected_elements:
            out[f"{prefix}_frac_{el}"] = 0.0
        for prop in ["Z", "atomic_mass", "X", "row", "group"]:
            for stat in ["mean", "min", "max", "range", "std"]:
                out[f"{prefix}_{prop}_{stat}"] = np.nan
        for flag in ["O", "F", "P", "S", "Si", "C", "N", "Cl", "Br", "I"]:
            out[f"{prefix}_has_{flag}"] = 0
        out[f"{prefix}_n_transition_metals"] = np.nan
        out[f"{prefix}_frac_transition_metals"] = np.nan
        return out

    el_amt = {str(el): float(amount) for el, amount in comp.get_el_amt_dict().items()}
    total_atoms = sum(el_amt.values())
    out[f"{prefix}_parse_ok"] = 1
    out[f"{prefix}_n_elements"] = len(el_amt)
    out[f"{prefix}_n_atoms"] = total_atoms
    for el in selected_elements:
        out[f"{prefix}_frac_{el}"] = el_amt.get(el, 0.0) / total_atoms if total_atoms > 0 else 0.0
    weights = []
    prop_values = {p: [] for p in ["Z", "atomic_mass", "X", "row", "group"]}
    for el, amt in el_amt.items():
        weights.append(amt)
        prop_values["Z"].append(element_property(el, "Z"))
        prop_values["atomic_mass"].append(element_property(el, "atomic_mass"))
        prop_values["X"].append(element_property(el, "X"))
        prop_values["row"].append(element_property(el, "row"))
        prop_values["group"].append(element_property(el, "group"))
    for prop, vals in prop_values.items():
        stats = weighted_stats(vals, weights)
        for stat, val in stats.items():
            out[f"{prefix}_{prop}_{stat}"] = val
    for flag in ["O", "F", "P", "S", "Si", "C", "N", "Cl", "Br", "I"]:
        out[f"{prefix}_has_{flag}"] = 1 if flag in el_amt else 0
    n_tm = sum(1 for el in el_amt if el in TRANSITION_METALS)
    tm_atoms = sum(amt for el, amt in el_amt.items() if el in TRANSITION_METALS)
    out[f"{prefix}_n_transition_metals"] = n_tm
    out[f"{prefix}_frac_transition_metals"] = tm_atoms / total_atoms if total_atoms > 0 else np.nan
    return out

def working_ion_features(ion: str):
    prefix = "wi"
    out = {
        f"{prefix}_is_Li": 1 if ion == "Li" else 0,
        f"{prefix}_is_Na": 1 if ion == "Na" else 0,
        f"{prefix}_is_K": 1 if ion == "K" else 0,
    }
    for prop in ["Z", "atomic_mass", "X", "row", "group"]:
        out[f"{prefix}_{prop}"] = element_property(ion, prop)
    return out

def formula_working_ion_amount(formula, ion: str):
    comp = parse_composition(formula)
    if comp is None:
        return np.nan
    try:
        return float(comp.get_el_amt_dict().get(ion, 0.0))
    except Exception:
        return np.nan

print(f"pymatgen available: {PYMATGEN_AVAILABLE}")

In [ ]:
# ============================================================
# Target plausibility masks
# ============================================================
# These masks do not delete records. They flag raw, physically plausible,
# and benchmark-useful subsets for Notebook 04.
# ============================================================

mask_df = core_df[[
    "record_index", "working_ion", "electrode_uid", "framework_uid", "coarse_family", "chemsys", "host_chemsys_no_working_ion",
]].copy()

mask_df["mask_raw_all_records"] = True
mask_df["mask_voltage_0_to_6"] = pd.to_numeric(core_df["average_voltage"], errors="coerce").between(0, 6, inclusive="both")
mask_df["mask_capacity_grav_positive_le_1000"] = pd.to_numeric(core_df["capacity_grav"], errors="coerce").between(0, 1000, inclusive="both")
mask_df["mask_capacity_vol_positive_le_10000"] = pd.to_numeric(core_df["capacity_vol"], errors="coerce").between(0, 10000, inclusive="both")
mask_df["mask_energy_grav_nonnegative_le_6000"] = pd.to_numeric(core_df["energy_grav"], errors="coerce").between(0, 6000, inclusive="both")
mask_df["mask_energy_vol_nonnegative_le_50000"] = pd.to_numeric(core_df["energy_vol"], errors="coerce").between(0, 50000, inclusive="both")
mask_df["mask_volume_change_0_to_2"] = pd.to_numeric(core_df["max_delta_volume"], errors="coerce").between(0, 2, inclusive="both")
mask_df["mask_stability_charge_0_to_2"] = pd.to_numeric(core_df["stability_charge"], errors="coerce").between(0, 2, inclusive="both")
mask_df["mask_stability_discharge_0_to_2"] = pd.to_numeric(core_df["stability_discharge"], errors="coerce").between(0, 2, inclusive="both")
mask_df["mask_stability_worst_0_to_2"] = pd.to_numeric(core_df["stability_worst"], errors="coerce").between(0, 2, inclusive="both")

mask_df["mask_physics_plausible_primary"] = mask_df[[
    "mask_voltage_0_to_6", "mask_capacity_grav_positive_le_1000", "mask_energy_grav_nonnegative_le_6000",
]].all(axis=1)

mask_df["mask_physics_plausible_all_targets"] = mask_df[[
    "mask_voltage_0_to_6", "mask_capacity_grav_positive_le_1000", "mask_capacity_vol_positive_le_10000",
    "mask_energy_grav_nonnegative_le_6000", "mask_energy_vol_nonnegative_le_50000", "mask_volume_change_0_to_2",
    "mask_stability_charge_0_to_2", "mask_stability_discharge_0_to_2", "mask_stability_worst_0_to_2",
]].all(axis=1)

mask_path = AUDIT_DIR / "02_target_plausibility_masks.csv"
mask_df.to_csv(mask_path, index=False)

mask_summary_rows = []
for col in mask_df.columns:
    if col.startswith("mask_"):
        mask_summary_rows.append({
            "mask": col,
            "n_pass": int(mask_df[col].sum()),
            "n_total": len(mask_df),
            "pass_pct": 100.0 * float(mask_df[col].sum()) / len(mask_df),
        })

mask_summary_df = pd.DataFrame(mask_summary_rows)
mask_summary_df.to_csv(AUDIT_DIR / "02_target_plausibility_summary.csv", index=False)

display(mask_summary_df)
print(f"Saved: {mask_path}")

In [ ]:
# ============================================================
# Build composition and stoichiometry feature block
# ============================================================

composition_feature_rows = []

for _, row in core_df.iterrows():
    d = {}
    # Clean composition source for P1
    d.update(formula_composition_features(row.get("framework_formula"), "fw"))
    # Generated for leakage audit only; these are NOT clean P1 features.
    d.update(formula_composition_features(row.get("formula_charge"), "charge_formula"))
    d.update(formula_composition_features(row.get("formula_discharge"), "discharge_formula"))
    # Working ion descriptors
    d.update(working_ion_features(safe_str(row.get("working_ion"))))

    ion = safe_str(row.get("working_ion"))
    wi_charge = formula_working_ion_amount(row.get("formula_charge"), ion)
    wi_discharge = formula_working_ion_amount(row.get("formula_discharge"), ion)

    d["stoich_wi_amount_charge"] = wi_charge
    d["stoich_wi_amount_discharge"] = wi_discharge
    if np.isfinite(wi_charge) and np.isfinite(wi_discharge):
        d["stoich_wi_amount_delta"] = wi_discharge - wi_charge
        d["stoich_abs_wi_amount_delta"] = abs(wi_discharge - wi_charge)
    else:
        d["stoich_wi_amount_delta"] = np.nan
        d["stoich_abs_wi_amount_delta"] = np.nan

    frac_charge = safe_float(row.get("fracA_charge"))
    frac_discharge = safe_float(row.get("fracA_discharge"))
    if np.isfinite(frac_charge) and np.isfinite(frac_discharge):
        d["stored_fracA_delta"] = frac_discharge - frac_charge
        d["stored_abs_fracA_delta"] = abs(frac_discharge - frac_charge)
    else:
        d["stored_fracA_delta"] = np.nan
        d["stored_abs_fracA_delta"] = np.nan

    composition_feature_rows.append(d)

composition_features_df = pd.DataFrame(composition_feature_rows, index=core_df.index)
feature_df = pd.concat([core_df.copy(), composition_features_df], axis=1)

print("Composition/stoichiometry features added.")
print(f"feature_df shape after composition block: {feature_df.shape}")
display(composition_features_df.head())

In [ ]:
# ============================================================
# Merge linked MP summary descriptors for charge/discharge states
# ============================================================
# These descriptors support P2/P3 protocol construction.
# Target-specific leakage rules below decide whether they are allowed.
# ============================================================

summary_clean = summary_df.copy()
summary_clean.columns = [sanitize_colname(c) for c in summary_clean.columns]

if "material_id" not in summary_clean.columns:
    raise ValueError("Linked summary table does not contain material_id after column sanitization.")

SUMMARY_KEEP_CANDIDATES = [
    "material_id", "formation_energy_per_atom", "is_metal", "volume", "last_updated", "theoretical",
    "chemsys", "energy_above_hull", "density", "formula_pretty", "nsites", "density_atomic",
    "band_gap", "is_stable", "symmetry_number", "symmetry_crystal_system",
]

summary_keep = [c for c in SUMMARY_KEEP_CANDIDATES if c in summary_clean.columns]
summary_small = summary_clean[summary_keep].drop_duplicates(subset=["material_id"]).copy()

def prefix_summary(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    rename_map = {c: f"{prefix}_{c}" for c in df.columns if c != "material_id"}
    return df.rename(columns=rename_map)

charge_summary = prefix_summary(summary_small, "charge_summary")
discharge_summary = prefix_summary(summary_small, "discharge_summary")

feature_df = feature_df.merge(charge_summary, left_on="id_charge", right_on="material_id", how="left").drop(columns=["material_id"], errors="ignore")
feature_df = feature_df.merge(discharge_summary, left_on="id_discharge", right_on="material_id", how="left").drop(columns=["material_id"], errors="ignore")

NUMERIC_SUMMARY_PATTERNS = [
    "summary_formation_energy_per_atom", "summary_energy_above_hull", "summary_volume", "summary_density",
    "summary_nsites", "summary_density_atomic", "summary_band_gap", "summary_symmetry_number",
]

for col in feature_df.columns:
    if any(pattern in col for pattern in NUMERIC_SUMMARY_PATTERNS):
        feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce")

BOOLEAN_SUMMARY_COLS = [
    "charge_summary_is_metal", "discharge_summary_is_metal", "charge_summary_is_stable", "discharge_summary_is_stable",
    "charge_summary_theoretical", "discharge_summary_theoretical",
]
for col in BOOLEAN_SUMMARY_COLS:
    if col in feature_df.columns:
        feature_df[col] = feature_df[col].map({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})

for role in ["charge_summary", "discharge_summary"]:
    vol_col = f"{role}_volume"
    nsites_col = f"{role}_nsites"
    if vol_col in feature_df.columns and nsites_col in feature_df.columns:
        feature_df[f"{role}_volume_per_site"] = feature_df[vol_col] / feature_df[nsites_col].replace(0, np.nan)

if "charge_summary_volume" in feature_df.columns and "discharge_summary_volume" in feature_df.columns:
    feature_df["summary_delta_volume_fraction"] = (
        feature_df["discharge_summary_volume"] - feature_df["charge_summary_volume"]
    ) / feature_df["charge_summary_volume"].replace(0, np.nan)
    feature_df["summary_abs_delta_volume_fraction"] = feature_df["summary_delta_volume_fraction"].abs()

for prop in ["formation_energy_per_atom", "energy_above_hull", "density", "density_atomic", "band_gap", "volume_per_site"]:
    c = f"charge_summary_{prop}"
    d = f"discharge_summary_{prop}"
    if c in feature_df.columns and d in feature_df.columns:
        feature_df[f"summary_delta_{prop}"] = feature_df[d] - feature_df[c]
        feature_df[f"summary_abs_delta_{prop}"] = (feature_df[d] - feature_df[c]).abs()

if "charge_summary_energy_above_hull" in feature_df.columns and "discharge_summary_energy_above_hull" in feature_df.columns:
    feature_df["summary_worst_energy_above_hull"] = feature_df[[
        "charge_summary_energy_above_hull", "discharge_summary_energy_above_hull"
    ]].max(axis=1)

print(f"feature_df shape after linked summary merge: {feature_df.shape}")

summary_merge_audit = pd.DataFrame([{
    "n_records": len(feature_df),
    "n_with_charge_summary": int(feature_df["charge_summary_formula_pretty"].notna().sum()) if "charge_summary_formula_pretty" in feature_df.columns else np.nan,
    "n_with_discharge_summary": int(feature_df["discharge_summary_formula_pretty"].notna().sum()) if "discharge_summary_formula_pretty" in feature_df.columns else np.nan,
}])
summary_merge_audit.to_csv(AUDIT_DIR / "02_summary_merge_audit.csv", index=False)
display(summary_merge_audit)

In [ ]:
# ============================================================
# Define candidate feature pool
# ============================================================

METADATA_COLS = {
    "record_index", "source_queried_working_ion", "source_endpoint", "mp_database_version", "record_source_id",
    "working_ion", "battery_formula", "formula_charge", "formula_discharge", "framework_formula", "chemsys",
    "elements", "id_charge", "id_discharge", "material_ids", "battery_type", "thermo_type", "last_updated",
    "framework_formula_reduced", "host_elements_no_working_ion", "host_chemsys_no_working_ion", "chemical_system_uid",
    "working_ion_group", "electrode_uid", "framework_uid", "coarse_family",
    "charge_summary_last_updated", "charge_summary_chemsys", "charge_summary_formula_pretty", "charge_summary_symmetry_crystal_system",
    "discharge_summary_last_updated", "discharge_summary_chemsys", "discharge_summary_formula_pretty", "discharge_summary_symmetry_crystal_system",
}

numeric_candidate_features = []

for col in feature_df.columns:
    if col in METADATA_COLS:
        continue
    converted = pd.to_numeric(feature_df[col], errors="coerce")
    if converted.notna().sum() > 0:
        feature_df[col] = converted
        numeric_candidate_features.append(col)

master_feature_path = PROCESSED_DIR / "02_master_feature_table.csv"
feature_df.to_csv(master_feature_path, index=False)

metadata_and_targets_cols = [c for c in [
    "record_index", "working_ion", "electrode_uid", "framework_uid", "chemical_system_uid",
    "host_chemsys_no_working_ion", "coarse_family", "battery_formula", "formula_charge",
    "formula_discharge", "framework_formula", "id_charge", "id_discharge",
] + TARGET_COLS if c in feature_df.columns]

metadata_targets_df = feature_df[metadata_and_targets_cols].copy()
metadata_targets_df.to_csv(PROCESSED_DIR / "02_master_metadata_and_targets.csv", index=False)

candidate_feature_pool_df = pd.DataFrame({
    "feature": numeric_candidate_features,
    "n_non_missing": [int(feature_df[f].notna().sum()) for f in numeric_candidate_features],
    "non_missing_pct": [100.0 * float(feature_df[f].notna().sum()) / len(feature_df) for f in numeric_candidate_features],
    "n_unique_values": [int(feature_df[f].nunique(dropna=True)) for f in numeric_candidate_features],
})

candidate_feature_pool_df.to_csv(AUDIT_DIR / "02_numeric_candidate_feature_pool.csv", index=False)

print(f"Master feature table saved: {master_feature_path}")
print(f"Number of numeric candidate features: {len(numeric_candidate_features)}")
display(candidate_feature_pool_df.head(20))

In [ ]:
# ============================================================
# Feature-origin and target-specific leakage rules — 
# ============================================================

LEAKAGE_LEVEL_RANK = {"L0": 0, "L1": 1, "L2": 2, "L3": 3, "L4": 4, "L5": 5}

P2_FORBIDDEN_SUBSTRINGS = [
    "formation_energy", "energy_above_hull", "band_gap", "is_metal", "is_stable", "theoretical",
]

P3_VOLTAGE_ENERGY_FORBIDDEN_SUBSTRINGS = ["formation_energy"]
P3_STABILITY_FORBIDDEN_SUBSTRINGS = ["energy_above_hull", "is_stable", "stability"]
P3_VOLUME_FORBIDDEN_SUBSTRINGS = ["volume", "density"]

def contains_any(feature: str, substrings: list[str]) -> bool:
    f = feature.lower()
    return any(s.lower() in f for s in substrings)

def feature_origin(feature: str) -> str:
    """
    Classify feature provenance/origin.
    This is separate from target-specific leakage level.

     behavior:
      - formation_energy / energy_above_hull / is_stable are post_dft_energy_stability_summary.
      - band_gap / is_metal are post_dft_electronic_summary.
      - theoretical is post_dft_provenance_flag.
      - volume/density/nsites/symmetry are relaxed_structure_summary.
    """
    f = feature.lower()

    if feature in TARGET_COLS:
        return "stored_electrode_target"
    if feature.startswith("fw_"):
        return "composition_known_framework"
    if feature.startswith("wi_is_"):
        return "domain_label_onehot"
    if feature.startswith("wi_"):
        return "composition_known_working_ion"
    if (
        feature.startswith("charge_formula_")
        or feature.startswith("discharge_formula_")
        or feature.startswith("stoich_")
        or feature.startswith("stored_fracA")
        or feature in ["fracA_charge", "fracA_discharge"]
    ):
        return "stoichiometric_window"
    if feature in ["num_steps", "max_voltage_step"]:
        return "stored_electrode_process_descriptor"

    if "summary" in f:
        if "formation_energy" in f or "energy_above_hull" in f or "is_stable" in f:
            return "post_dft_energy_stability_summary"
        if "band_gap" in f or "is_metal" in f:
            return "post_dft_electronic_summary"
        if "theoretical" in f:
            return "post_dft_provenance_flag"
        if any(s in f for s in ["volume", "density", "nsites", "symmetry"]):
            return "relaxed_structure_summary"
        return "post_dft_other_summary"

    return "other_numeric"

def target_specific_leakage(feature: str, target: str):
    """
    Return: leakage_level, leakage_reason.

     conservative rules:
      - P2-relevant post-DFT energy/electronic/provenance features are not treated as relaxed-structure clean.
      - voltage/energy targets treat endpoint formation-energy descriptors as L2.
      - stability targets treat endpoint hull/is_stable descriptors as L2.
      - volume-change target treats endpoint volume/density descriptors as L2.
    """
    origin = feature_origin(feature)
    f = feature.lower()

    if feature == target:
        return "L1", "direct duplicate of the target column"

    if origin == "domain_label_onehot":
        return "L5", "working-ion one-hot/domain label; use only for domain stress tests"

    energy_targets = {"energy_grav", "energy_vol"}
    capacity_targets = {"capacity_grav", "capacity_vol"}
    stability_targets = {"stability_charge", "stability_discharge", "stability_worst"}

    # All other stored targets are post-hoc computed electrode properties.
    if feature in TARGET_COLS:
        if target in energy_targets and feature in ["average_voltage", "capacity_grav", "capacity_vol", "energy_grav", "energy_vol"]:
            return "L2", "energy is target-defined by voltage/capacity and unit-related energy/capacity columns"
        if target in capacity_targets and feature in ["capacity_grav", "capacity_vol", "energy_grav", "energy_vol"]:
            return "L2", "capacity/energy variants are mathematically target-adjacent"
        if target == "average_voltage" and feature in ["energy_grav", "energy_vol"]:
            return "L2", "energy combines voltage with capacity"
        if target in stability_targets and feature in stability_targets:
            return "L2", "stored endpoint/worst stability fields are target-defining or target-adjacent"
        if target == "max_delta_volume" and feature == "max_delta_volume":
            return "L1", "direct target duplicate"
        return "L4", "other stored electrode target/property; post hoc computed-record descriptor"

    # Stoichiometric insertion descriptors.
    if target in energy_targets or target in capacity_targets:
        if origin == "stoichiometric_window":
            return "L3", "working-ion insertion stoichiometry directly controls theoretical capacity/energy"

    # Energy targets.
    if target in energy_targets:
        if feature in ["average_voltage", "capacity_grav", "capacity_vol", "energy_grav", "energy_vol"] and feature != target:
            return "L2", "energy is target-defined by voltage and capacity and unit-related target columns"
        if "formation_energy" in f:
            return "L2", "endpoint formation-energy descriptor is target-adjacent/target-defining for computed electrode energy"
        if origin in {"post_dft_energy_stability_summary", "post_dft_electronic_summary", "post_dft_other_summary"}:
            return "L4", "post-DFT descriptor adjacent to computed electrode energy"

    # Capacity targets.
    if target in capacity_targets:
        if feature in ["capacity_grav", "capacity_vol", "energy_grav", "energy_vol"] and feature != target:
            return "L2", "capacity/energy variants are mathematically target-adjacent"
        if origin in {"post_dft_energy_stability_summary", "post_dft_electronic_summary", "post_dft_other_summary"}:
            return "L4", "post-DFT descriptor; decision-support only, not clean pre-DFT capacity prediction"

    # Voltage target.
    if target == "average_voltage":
        if feature in ["energy_grav", "energy_vol"]:
            return "L2", "energy combines voltage with capacity"
        if feature == "max_voltage_step":
            return "L4", "stored voltage-window descriptor adjacent to voltage target"
        if "formation_energy" in f:
            return "L2", "endpoint formation-energy descriptor can reconstruct computed insertion voltage trends"
        if origin in {"post_dft_energy_stability_summary", "post_dft_electronic_summary", "post_dft_other_summary"}:
            return "L4", "post-DFT descriptor adjacent to computed voltage"

    # Stability targets.
    if target in stability_targets:
        if (
            "energy_above_hull" in f
            or "is_stable" in f
            or feature in ["stability_charge", "stability_discharge", "stability_worst"]
        ):
            return "L2", "endpoint hull/stability descriptor is target-defining for stability target"
        if "formation_energy" in f:
            return "L4", "post-DFT formation-energy descriptor adjacent to stability but not direct hull target"
        if origin in {"post_dft_energy_stability_summary", "post_dft_electronic_summary", "post_dft_other_summary"}:
            return "L4", "post-DFT descriptor adjacent to computed stability"

    # Volume-change target.
    if target == "max_delta_volume":
        if "volume" in f or "density" in f:
            return "L2", "endpoint volume/density descriptors define or strongly encode volume-change target"
        if origin in {"post_dft_energy_stability_summary", "post_dft_electronic_summary", "post_dft_other_summary"}:
            return "L4", "post-DFT descriptor adjacent to computed electrode record"

    # Generic post-DFT summaries.
    if origin in {"post_dft_energy_stability_summary", "post_dft_electronic_summary", "post_dft_provenance_flag", "post_dft_other_summary"}:
        return "L4", "post-DFT summary descriptor; decision-support only"

    return "L0", "no direct target relationship identified by rule-based audit"

def p3_target_forbidden(feature: str, target: str) -> bool:
    """Explicit target-specific P3 exclusions beyond leakage-level logic."""
    if target in {"average_voltage", "energy_grav", "energy_vol"}:
        return contains_any(feature, P3_VOLTAGE_ENERGY_FORBIDDEN_SUBSTRINGS)
    if target in {"stability_charge", "stability_discharge", "stability_worst"}:
        return contains_any(feature, P3_STABILITY_FORBIDDEN_SUBSTRINGS)
    if target == "max_delta_volume":
        return contains_any(feature, P3_VOLUME_FORBIDDEN_SUBSTRINGS)
    return False

def allowed_protocol(feature: str, target: str, protocol: str) -> bool:
    leakage_level, _ = target_specific_leakage(feature, target)
    origin = feature_origin(feature)

    if protocol == "P4":
        # Stress test allows direct leakage except group/domain leakage.
        return leakage_level != "L5"

    if protocol == "P0":
        # Leaky full-feature baseline, but direct target duplicate and group/domain leakage are removed.
        return leakage_level not in {"L1", "L5"}

    if protocol == "P1":
        return origin in {"composition_known_framework", "composition_known_working_ion"} and leakage_level == "L0"

    if protocol == "P2":
        if contains_any(feature, P2_FORBIDDEN_SUBSTRINGS):
            return False
        return origin in {
            "composition_known_framework",
            "composition_known_working_ion",
            "relaxed_structure_summary",
        } and leakage_level == "L0"

    if protocol == "P3":
        if p3_target_forbidden(feature, target):
            return False
        return origin in {
            "composition_known_framework",
            "composition_known_working_ion",
            "relaxed_structure_summary",
            "post_dft_energy_stability_summary",
            "post_dft_electronic_summary",
            "post_dft_other_summary",
            "stored_electrode_process_descriptor",
        } and leakage_level not in {"L1", "L2", "L3", "L5"}

    return False

print(" leakage rule functions defined.")
print("P2 forbidden substrings:", P2_FORBIDDEN_SUBSTRINGS)
print("P3 voltage/energy forbidden substrings:", P3_VOLTAGE_ENERGY_FORBIDDEN_SUBSTRINGS)
print("P3 stability forbidden substrings:", P3_STABILITY_FORBIDDEN_SUBSTRINGS)
print("P3 volume forbidden substrings:", P3_VOLUME_FORBIDDEN_SUBSTRINGS)

In [ ]:
# ============================================================
# Feature leakage taxonomy master table
# ============================================================

taxonomy_rows = []

for feature in numeric_candidate_features:
    target_levels = [target_specific_leakage(feature, target)[0] for target in TARGET_COLS]
    max_level = max(target_levels, key=lambda x: LEAKAGE_LEVEL_RANK.get(x, 0))
    taxonomy_rows.append({
        "feature": feature,
        "feature_origin": feature_origin(feature),
        "default_max_leakage_level_across_targets": max_level,
        "is_target_column": feature in TARGET_COLS,
        "n_non_missing": int(feature_df[feature].notna().sum()),
        "non_missing_pct": 100.0 * float(feature_df[feature].notna().sum()) / len(feature_df),
        "n_unique_values": int(feature_df[feature].nunique(dropna=True)),
    })

feature_taxonomy_master_df = pd.DataFrame(taxonomy_rows).sort_values(
    ["default_max_leakage_level_across_targets", "feature_origin", "feature"]
)

taxonomy_path = AUDIT_DIR / "02_feature_leakage_taxonomy_master.csv"
feature_taxonomy_master_df.to_csv(taxonomy_path, index=False)

taxonomy_summary_df = (
    feature_taxonomy_master_df
    .groupby(["default_max_leakage_level_across_targets", "feature_origin"], dropna=False)
    .size()
    .reset_index(name="n_features")
    .sort_values(["default_max_leakage_level_across_targets", "feature_origin"])
)
taxonomy_summary_df.to_csv(AUDIT_DIR / "02_feature_taxonomy_summary.csv", index=False)

display(feature_taxonomy_master_df.head(30))
display(taxonomy_summary_df)
print(f"Saved: {taxonomy_path}")

In [ ]:
# ============================================================
# Target-specific leakage matrix
# ============================================================

matrix_rows = []

for target in TARGET_COLS:
    for feature in numeric_candidate_features:
        leakage_level, leakage_reason = target_specific_leakage(feature, target)
        row = {
            "target": target,
            "target_unit": TARGET_UNITS.get(target, ""),
            "feature": feature,
            "feature_origin": feature_origin(feature),
            "leakage_level": leakage_level,
            "leakage_reason": leakage_reason,
            "p3_explicit_forbidden": p3_target_forbidden(feature, target),
            "p2_forbidden_keyword": contains_any(feature, P2_FORBIDDEN_SUBSTRINGS),
        }
        for protocol in ["P0", "P1", "P2", "P3", "P4"]:
            row[f"allowed_{protocol}"] = allowed_protocol(feature, target, protocol)
        matrix_rows.append(row)

target_specific_leakage_matrix_df = pd.DataFrame(matrix_rows)

matrix_path = AUDIT_DIR / "02_target_specific_leakage_matrix.csv"
target_specific_leakage_matrix_df.to_csv(matrix_path, index=False)

leakage_matrix_summary_df = (
    target_specific_leakage_matrix_df
    .groupby(["target", "leakage_level"], dropna=False)
    .size()
    .reset_index(name="n_features")
    .sort_values(["target", "leakage_level"])
)
leakage_matrix_summary_df.to_csv(AUDIT_DIR / "02_target_specific_leakage_summary.csv", index=False)

display(leakage_matrix_summary_df)
print(f"Saved: {matrix_path}")

In [ ]:
# ============================================================
# Build target-specific protocol feature lists
# ============================================================

protocol_summary_rows = []

for target in TARGET_COLS:
    for protocol in ["P0", "P1", "P2", "P3", "P4"]:
        allowed_col = f"allowed_{protocol}"
        feature_list = target_specific_leakage_matrix_df.loc[
            (target_specific_leakage_matrix_df["target"] == target)
            & (target_specific_leakage_matrix_df[allowed_col]),
            "feature",
        ].tolist()

        list_df = pd.DataFrame({"target": target, "protocol": protocol, "feature": feature_list})

        list_csv_path = PROTOCOL_DIR / f"02_protocol_{protocol}_features_for_target_{target}.csv"
        list_json_path = PROTOCOL_DIR / f"02_protocol_{protocol}_features_for_target_{target}.json"

        list_df.to_csv(list_csv_path, index=False)
        write_json_safe({
            "target": target,
            "target_unit": TARGET_UNITS.get(target, ""),
            "protocol": protocol,
            "protocol_name": PROTOCOL_DEFINITIONS[protocol]["name"],
            "protocol_description": PROTOCOL_DEFINITIONS[protocol]["description"],
            "n_features": len(feature_list),
            "features": feature_list,
        }, list_json_path)

        sub = target_specific_leakage_matrix_df[
            (target_specific_leakage_matrix_df["target"] == target)
            & (target_specific_leakage_matrix_df["feature"].isin(feature_list))
        ]
        level_counts = sub["leakage_level"].value_counts().to_dict()
        origin_counts = sub["feature_origin"].value_counts().to_dict()

        protocol_summary_rows.append({
            "target": target,
            "target_unit": TARGET_UNITS.get(target, ""),
            "protocol": protocol,
            "protocol_name": PROTOCOL_DEFINITIONS[protocol]["name"],
            "n_features": len(feature_list),
            "n_L0": int(level_counts.get("L0", 0)),
            "n_L1": int(level_counts.get("L1", 0)),
            "n_L2": int(level_counts.get("L2", 0)),
            "n_L3": int(level_counts.get("L3", 0)),
            "n_L4": int(level_counts.get("L4", 0)),
            "n_L5": int(level_counts.get("L5", 0)),
            "feature_origin_counts_json": json.dumps(origin_counts, sort_keys=True),
            "feature_list_csv": str(list_csv_path),
            "feature_list_json": str(list_json_path),
        })

protocol_summary_df = pd.DataFrame(protocol_summary_rows)
protocol_summary_path = AUDIT_DIR / "02_protocol_feature_counts_by_target.csv"
protocol_summary_df.to_csv(protocol_summary_path, index=False)

display(protocol_summary_df)
print(f"Saved: {protocol_summary_path}")
print(f"Feature lists saved under: {PROTOCOL_DIR}")

In [ ]:
# ============================================================
# Protocol integrity audit —  AND STRICTER
# ============================================================

integrity_rows = []

def load_protocol_features(target: str, protocol: str) -> list[str]:
    p = PROTOCOL_DIR / f"02_protocol_{protocol}_features_for_target_{target}.csv"
    if not p.exists():
        return []
    return pd.read_csv(p)["feature"].astype(str).tolist()

for _, row in protocol_summary_df.iterrows():
    target = row["target"]
    protocol = row["protocol"]
    features = load_protocol_features(target, protocol)

    status = "PASS"
    issues = []

    # Clean protocols must not contain L1, L2, L3, L4, or L5.
    if protocol in ["P1", "P2"]:
        for level in ["L1", "L2", "L3", "L4", "L5"]:
            if int(row[f"n_{level}"]) > 0:
                status = "FAIL"
                issues.append(f"{protocol} contains {level} features")

    # PATCH: P2 must not contain post-DFT energy/electronic/provenance fields.
    if protocol == "P2":
        for forbidden in P2_FORBIDDEN_SUBSTRINGS:
            bad = [f for f in features if forbidden.lower() in f.lower()]
            if bad:
                status = "FAIL"
                issues.append(f"P2 contains forbidden substring '{forbidden}': {bad[:5]}")

    # P3 may contain L4, but not L1/L2/L3/L5.
    if protocol == "P3":
        for level in ["L1", "L2", "L3", "L5"]:
            if int(row[f"n_{level}"]) > 0:
                status = "FAIL"
                issues.append(f"P3 contains {level} features")

        # PATCH: explicit target-specific P3 bans.
        if target in {"average_voltage", "energy_grav", "energy_vol"}:
            bad = [f for f in features if contains_any(f, P3_VOLTAGE_ENERGY_FORBIDDEN_SUBSTRINGS)]
            if bad:
                status = "FAIL"
                issues.append(f"P3 voltage/energy contains forbidden formation-energy features: {bad[:5]}")

        if target in {"stability_charge", "stability_discharge", "stability_worst"}:
            bad = [f for f in features if contains_any(f, P3_STABILITY_FORBIDDEN_SUBSTRINGS)]
            if bad:
                status = "FAIL"
                issues.append(f"P3 stability contains forbidden hull/stability features: {bad[:5]}")

        if target == "max_delta_volume":
            bad = [f for f in features if contains_any(f, P3_VOLUME_FORBIDDEN_SUBSTRINGS)]
            if bad:
                status = "FAIL"
                issues.append(f"P3 volume-change contains forbidden volume/density features: {bad[:5]}")

    # P0 should not contain direct target duplicates or domain leakage.
    if protocol == "P0":
        for level in ["L1", "L5"]:
            if int(row[f"n_{level}"]) > 0:
                status = "FAIL"
                issues.append(f"P0 contains {level} features")

    # P4 intentionally leaks but should not contain L5 and should contain L1.
    if protocol == "P4":
        if int(row["n_L5"]) > 0:
            status = "FAIL"
            issues.append("P4 contains L5 group/domain leakage")
        if int(row["n_L1"]) == 0:
            status = "WARNING" if status == "PASS" else status
            issues.append("P4 does not contain L1 direct target duplicate; stress test may be weak")

    if int(row["n_features"]) == 0:
        status = "FAIL"
        issues.append("zero features")

    integrity_rows.append({
        "target": target,
        "protocol": protocol,
        "status": status,
        "n_features": int(row["n_features"]),
        "issues": "; ".join(issues) if issues else "none",
    })

protocol_integrity_df = pd.DataFrame(integrity_rows)
integrity_path = AUDIT_DIR / "02_protocol_integrity_audit.csv"
protocol_integrity_df.to_csv(integrity_path, index=False)

display(protocol_integrity_df)

n_fail = int((protocol_integrity_df["status"] == "FAIL").sum())
n_warning = int((protocol_integrity_df["status"] == "WARNING").sum())

if n_fail > 0:
    log_event("protocol_integrity", "ERROR", "Protocol integrity audit contains FAIL rows.", {"n_fail": n_fail, "n_warning": n_warning})
else:
    log_event("protocol_integrity", "INFO", "Protocol integrity audit passed with no FAIL rows.", {"n_warning": n_warning})

print(f"Protocol integrity FAIL rows: {n_fail}")
print(f"Protocol integrity WARNING rows: {n_warning}")
save_event_log()

In [ ]:
# ============================================================
# Save protocol-ready manifest for Notebook 04
# ============================================================

protocol_manifest = {
    "notebook": "02_leakage_taxonomy_and_descriptor_protocols.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "input_notebook_01_dir": str(NB08_DIR),
    "output_notebook_02_dir": str(BASE_DIR),
    "master_feature_table": str(PROCESSED_DIR / "02_master_feature_table.csv"),
    "metadata_and_targets_table": str(PROCESSED_DIR / "02_master_metadata_and_targets.csv"),
    "target_plausibility_masks": str(AUDIT_DIR / "02_target_plausibility_masks.csv"),
    "feature_taxonomy_master": str(AUDIT_DIR / "02_feature_leakage_taxonomy_master.csv"),
    "target_specific_leakage_matrix": str(AUDIT_DIR / "02_target_specific_leakage_matrix.csv"),
    "protocol_feature_counts": str(AUDIT_DIR / "02_protocol_feature_counts_by_target.csv"),
    "protocol_integrity_audit": str(AUDIT_DIR / "02_protocol_integrity_audit.csv"),
    "protocol_feature_list_dir": str(PROTOCOL_DIR),
    "targets": TARGET_COLS,
    "primary_benchmark_targets": PRIMARY_BENCHMARK_TARGETS,
    "protocols": list(PROTOCOL_DEFINITIONS.keys()),
    "patch_notes": [
        "P2 excludes post-DFT energy/electronic/provenance features.",
        "P3 applies explicit target-specific bans for voltage/energy, stability, and volume-change targets.",
        "Protocol integrity audit includes explicit P2/P3 forbidden-feature checks.",
    ],
    "no_ml_training_performed": True,
    "no_ranking_performed": True,
    "no_cde_matching_performed": True,
    "no_criticality_filtering_performed": True,
}

write_json_safe(protocol_manifest, METADATA_DIR / "02_protocol_ready_manifest.json")

protocol_lookup_rows = []
for target in TARGET_COLS:
    for protocol in ["P0", "P1", "P2", "P3", "P4"]:
        protocol_lookup_rows.append({
            "target": target,
            "protocol": protocol,
            "feature_list_csv": str(PROTOCOL_DIR / f"02_protocol_{protocol}_features_for_target_{target}.csv"),
            "feature_list_json": str(PROTOCOL_DIR / f"02_protocol_{protocol}_features_for_target_{target}.json"),
            "master_feature_table": str(PROCESSED_DIR / "02_master_feature_table.csv"),
            "mask_table": str(AUDIT_DIR / "02_target_plausibility_masks.csv"),
        })

protocol_lookup_df = pd.DataFrame(protocol_lookup_rows)
protocol_lookup_df.to_csv(PROCESSED_DIR / "02_protocol_lookup_for_notebook_10.csv", index=False)

display(protocol_lookup_df.head(20))
print("Saved Notebook 04 protocol lookup table.")

In [ ]:
# ============================================================
# Final output manifest and decision
# ============================================================

def list_output_files(base_dir: Path):
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            rows.append({
                "relative_path": str(path.relative_to(base_dir)),
                "size_bytes": path.stat().st_size,
                "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
    return pd.DataFrame(rows)

output_manifest_df = list_output_files(BASE_DIR)
output_manifest_df.to_csv(METADATA_DIR / "02_output_file_manifest.csv", index=False)

n_integrity_fail = int((protocol_integrity_df["status"] == "FAIL").sum())
n_integrity_warning = int((protocol_integrity_df["status"] == "WARNING").sum())

required_outputs = [
    PROCESSED_DIR / "02_master_feature_table.csv",
    PROCESSED_DIR / "02_master_metadata_and_targets.csv",
    PROCESSED_DIR / "02_protocol_lookup_for_notebook_10.csv",
    AUDIT_DIR / "02_target_plausibility_masks.csv",
    AUDIT_DIR / "02_feature_leakage_taxonomy_master.csv",
    AUDIT_DIR / "02_target_specific_leakage_matrix.csv",
    AUDIT_DIR / "02_protocol_feature_counts_by_target.csv",
    AUDIT_DIR / "02_protocol_integrity_audit.csv",
    METADATA_DIR / "02_protocol_ready_manifest.json",
    METADATA_DIR / "02_no_ml_assertion.json",
]

missing_required_outputs = [str(p) for p in required_outputs if not p.exists()]

if missing_required_outputs:
    FINAL_DECISION_09 = "NO_GO_FIX_NOTEBOOK_02_OUTPUTS"
elif n_integrity_fail > 0:
    FINAL_DECISION_09 = "NO_GO_FIX_PROTOCOL_INTEGRITY"
elif n_integrity_warning > 0:
    FINAL_DECISION_09 = "CONDITIONAL_GO_TO_NOTEBOOK_04_REVIEW_WARNINGS"
else:
    FINAL_DECISION_09 = "FULL_GO_TO_NOTEBOOK_10"

final_decision_09 = {
    "final_decision": FINAL_DECISION_09,
    "n_integrity_fail": n_integrity_fail,
    "n_integrity_warning": n_integrity_warning,
    "missing_required_outputs": missing_required_outputs,
    "patch_status": "P2/P3 leakage rules tightened",
    "no_ml_training_performed": True,
    "no_ranking_performed": True,
    "no_cde_matching_performed": True,
    "no_criticality_filtering_performed": True,
}

write_json_safe(final_decision_09, METADATA_DIR / "02_final_decision.json")

display(output_manifest_df)

print("\n" + "=" * 80)
print(f"Notebook 02  FINAL DECISION: {FINAL_DECISION_09}")
print("=" * 80)

print("\nKey outputs:")
for p in required_outputs:
    print(f" - {p}  {'[OK]' if p.exists() else '[MISSING]'}")

save_event_log()
print("\nNotebook 02 complete.")